## Set-up

In [6]:
import wbdata as wb
import requests as rq
import pandas as pd

## World Bank API

In [7]:
countries = ["CA", "FR", "DE", "IT", "JP", "GB", "US"]

indicators = {
    "FP.CPI.TOTL.ZG": "CPI_inflation",
    "NY.GDP.MKTP.KD.ZG": "GDP_growth",
    "SL.UEM.TOTL.ZS": "Unemployment",
    "FM.LBL.BMNY.ZG": "Money_growth",
    "NE.CON.GOVT.ZS": "Government_consumption",
    "PA.NUS.FCRF": "Exchange_rates",
    "NE.IMP.GNFS.ZS": "Imports_%"
}

wb_df = wb.get_dataframe(indicators, country = countries)

# date is stored as index

wb_df = wb_df.reset_index()

wb_df = wb_df.rename(columns={"date": "Year", "country":"Country"})

wb_df["Year"] = pd.to_numeric(wb_df["Year"])


In [8]:
wb_df.head()
wb_df.info()

<class 'wbdata.client.DataFrame'>
RangeIndex: 462 entries, 0 to 461
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Country                 462 non-null    object 
 1   Year                    462 non-null    int64  
 2   CPI_inflation           461 non-null    float64
 3   GDP_growth              455 non-null    float64
 4   Unemployment            245 non-null    float64
 5   Money_growth            242 non-null    float64
 6   Government_consumption  419 non-null    float64
 7   Exchange_rates          462 non-null    float64
 8   Imports_%               409 non-null    float64
dtypes: float64(7), int64(1), object(1)
memory usage: 32.6+ KB


## OECD - Housing

In [9]:

# Housing Index

path_houses = "data/OECD.ECO.MPD,DSD_AN_HOUSE_PRICES@DF_HOUSE_PRICES,1.0+all.csv"

df_houses = pd.read_csv(path_houses)

df_houses 

# Select only rows of G7 countries - Check reference areas

df_houses["REF_AREA"].unique()

df_houses = df_houses[df_houses["REF_AREA"].isin(["ITA", "DEU", "CAN", "FRA", "JPN", "USA", "GBR"])]

# check all relevant countries have been selected

df_houses["Reference area"].unique()

# select annual data - not quarterly

df_houses["FREQ"].unique()

df_houses = df_houses[df_houses["FREQ"] == "A"]

# rename relevant columns

df_houses = df_houses.rename(columns = {"Reference area":"Country", "OBS_VALUE":"HPI", "TIME_PERIOD":"Year"})

# look at a specific example to see duplicates

canada_2000 = df_houses[ (df_houses["Country"] == "Canada") & (df_houses["Year"] == "2000") ]

# different measures - pick HPI (nominal house price indices)

df_houses = df_houses[df_houses["MEASURE"] == "HPI"]

# select only relevant columns

df_houses = df_houses[["Country", "HPI", "Year"]]

# create new column for percentage change - group by country and make sure years are in ascending order

df_houses["HPI"] = pd.to_numeric(df_houses["HPI"])

df_houses = df_houses.sort_values(by="Year")

df_houses["HPI_growth"] = df_houses.groupby("Country")["HPI"].pct_change()*100

df_houses["Year"] = pd.to_numeric(df_houses["Year"])

In [10]:
df_houses.head()
df_houses.info()

<class 'pandas.core.frame.DataFrame'>
Index: 403 entries, 41678 to 41498
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Country     403 non-null    object 
 1   HPI         403 non-null    float64
 2   Year        403 non-null    int64  
 3   HPI_growth  396 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 15.7+ KB


## OECD - Average Annual Wage (USD_PPP)

In [11]:

path_wage = "data/OECD.ELS.SAE,DSD_EARNINGS@AV_AN_WAGE,1.0+all.csv"

df_wage = pd.read_csv(path_wage)

df_wage = df_wage[df_wage["REF_AREA"].isin(["ITA", "DEU", "CAN", "FRA", "JPN", "USA", "GBR"])]

df_wage = df_wage.rename(columns = {"Reference area":"Country", "OBS_VALUE":"Average_wage_USD_PPP", "TIME_PERIOD":"Year"})

# look at specific example to see duplicates

canada_2000 = df_wage[ (df_wage["Country"] == "Canada") & (df_wage["Year"] == 2000) ]

# pick unit of measure - 3 types

df_wage = df_wage[df_wage["UNIT_MEASURE"] == "USD_PPP"]

df_wage = df_wage[["Country", "Year", "Average_wage_USD_PPP"]]

# create new column showing percentage change in annual wage

df_wage = df_wage.sort_values(by="Year")
 
df_wage["Wage_growth"] =  df_wage.groupby("Country")["Average_wage_USD_PPP"].pct_change()*100

df_wage["Year"] = pd.to_numeric(df_wage["Year"])

In [12]:
df_wage.head()
df_wage.info()

<class 'pandas.core.frame.DataFrame'>
Index: 252 entries, 108 to 1296
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Country               252 non-null    object 
 1   Year                  252 non-null    int64  
 2   Average_wage_USD_PPP  252 non-null    float64
 3   Wage_growth           245 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 9.8+ KB


## OECD - Interest Rates

In [23]:

path_interest = "data/OECD.SDD.STES,DSD_STES@DF_FINMARK,+all.csv"

df_interest = pd.read_csv(path_interest)

df_interest = df_interest[df_interest["REF_AREA"].isin(["ITA", "DEU", "CAN", "FRA", "JPN", "USA", "GBR"])]

# original file is too large 
merged_df.to_csv("cleaned_data/OECD.SDD.STES,DSD_STES@DF_FINMARK,+all.csv", index=False)

df_interest["FREQ"].unique()

df_interest = df_interest[df_interest["FREQ"] == "A"]

df_interest = df_interest.rename(columns = {"Reference area":"Country", "OBS_VALUE":"Interest_rate", "TIME_PERIOD":"Year"})

# canada example again

canada_2000 = df_interest[ (df_interest["Country"] == "Canada") & (df_interest["Year"] == "2000") ]

# 7 units - pick short term interest

df_interest = df_interest[df_interest["MEASURE"] == "IR3TIB"]

df_interest = df_interest[["Country", "Year", "Interest_rate"]]

df_interest["Year"] = pd.to_numeric(df_interest["Year"])

In [14]:
df_interest.head()
df_interest.info()

<class 'pandas.core.frame.DataFrame'>
Index: 362 entries, 164068 to 172236
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Country        362 non-null    object 
 1   Year           362 non-null    int64  
 2   Interest_rate  362 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 11.3+ KB


## World Bank Commodities - Oil Prices

In [15]:
path_commodities = "data/CMO-Historical-Data-Annual.csv"

df_commodities = pd.read_csv(path_commodities, skiprows= 7)

# select onyl "Crude oil, average" and Year

df_commodities = df_commodities.iloc[:, [0,1]]

# rename column labels

df_commodities.columns = ["Year", "Crude_oil_$/bbl"]

# remove columns after 2025

df_commodities = df_commodities.iloc[:66]

# create new column showing percentage change in oil price
 
df_commodities["Crude_Oil_Change"] =  df_commodities["Crude_oil_$/bbl"].pct_change()*100

df_commodities["Year"] = pd.to_numeric(df_commodities["Year"])

In [16]:
df_commodities.head()
df_commodities.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Year              66 non-null     float64
 1   Crude_oil_$/bbl   66 non-null     float64
 2   Crude_Oil_Change  65 non-null     float64
dtypes: float64(3)
memory usage: 1.7 KB


## Merging data

In [17]:
merged_df = pd.merge(wb_df, df_houses, on=["Country", "Year"], how= "left")

merged_df = pd.merge(merged_df, df_wage, on=["Country", "Year"], how= "left")

merged_df = pd.merge(merged_df, df_interest, on=["Country", "Year"], how= "left")

merged_df = pd.merge(merged_df, df_commodities, on="Year", how= "left")

In [18]:
merged_df.head()
merged_df.info()

<class 'wbdata.client.DataFrame'>
RangeIndex: 462 entries, 0 to 461
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Country                 462 non-null    object 
 1   Year                    462 non-null    int64  
 2   CPI_inflation           461 non-null    float64
 3   GDP_growth              455 non-null    float64
 4   Unemployment            245 non-null    float64
 5   Money_growth            242 non-null    float64
 6   Government_consumption  419 non-null    float64
 7   Exchange_rates          462 non-null    float64
 8   Imports_%               409 non-null    float64
 9   HPI                     403 non-null    float64
 10  HPI_growth              396 non-null    float64
 11  Average_wage_USD_PPP    252 non-null    float64
 12  Wage_growth             245 non-null    float64
 13  Interest_rate           358 non-null    float64
 14  Crude_oil_$/bbl         462 non-null    float6

## Data Validation

In [19]:
merged_df.duplicated(["Country", "Year"]).sum()

merged_df.isna().sum()

Country                     0
Year                        0
CPI_inflation               1
GDP_growth                  7
Unemployment              217
Money_growth              220
Government_consumption     43
Exchange_rates              0
Imports_%                  53
HPI                        59
HPI_growth                 66
Average_wage_USD_PPP      210
Wage_growth               217
Interest_rate             104
Crude_oil_$/bbl             0
Crude_Oil_Change            7
dtype: int64

## Save data frame

In [21]:
merged_df.to_csv("data/panel_data.csv", index=False)